# Speech Denoising — Dilated CNN

Dilated 1D CNN with exponentially growing receptive field.

**Architecture:** Conv1d with dilation=1,2,4,8 | **Loss:** MSELoss | **Optimizer:** Adam | **Epochs:** 4

**Motivation:** Standard CNN with kernel=3 and 4 layers sees only 9 samples (0.0005s). Dilated convolutions expand the receptive field to 31 samples without increasing parameters — each layer captures patterns at a different time scale.

---

## 1. Imports

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm

## 2. Load Preprocessed Data

In [ ]:
train_noisy_chunks = np.load('data/train_noisy_chunks.npy')
train_clean_chunks = np.load('data/train_clean_chunks.npy')

In [ ]:
torch_train_noisy_chunks = torch.from_numpy(train_noisy_chunks)
torch_train_clean_chunks = torch.from_numpy(train_clean_chunks)

## 3. Dataset Split (80/20 train/val)

In [ ]:
full_dataset = torch.utils.data.TensorDataset(torch_train_noisy_chunks, torch_train_clean_chunks)

In [ ]:
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32,)
val_dataloader = DataLoader(val_dataset, batch_size=32,)

## 4. Model Architecture

**Dilated Conv1d** — each layer uses increasing dilation to capture longer-range dependencies:
- layer1: dilation=1, padding=1 — local patterns (3 samples)
- layer2: dilation=2, padding=2 — medium patterns (5 samples)
- layer3: dilation=4, padding=4 — wider patterns (9 samples)
- layer4: dilation=8, padding=8 — long-range patterns (17 samples)

Total receptive field: ~31 samples vs 9 in standard CNN.
padding = dilation * (kernel_size - 1) / 2 to preserve sequence length.

In [ ]:
class DenoisingModelDilated(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, kernel_size=3, dilation=1, padding=1)
        self.layer2 = nn.Conv1d(16, 32, kernel_size=3, dilation=2, padding=2)
        self.layer3 = nn.Conv1d(32, 16, kernel_size=3, dilation=4, padding=4)
        self.layer4 = nn.Conv1d(16, 1, kernel_size=3, dilation=8, padding=8)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [ ]:
model = DenoisingModelDilated()
print(model)

## 5. Loss Function & Optimizer

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001 )

## 6. Training Loop with Validation

In [ ]:
n_epochs = 4

for epoch in tqdm(range(n_epochs)):
    epoch_loss = 0
    for batch in train_dataloader:
        noisy_batch, clean_batch = batch
        optimizer.zero_grad()
        train_pred_clean = model(noisy_batch)
        train_loss = loss_fn(train_pred_clean, clean_batch)
        train_loss.backward()
        optimizer.step()
        epoch_loss += train_loss.item()
    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch}, Avg Loss: {avg_epoch_loss:.4f}")
    
    model.eval()
    with torch.no_grad():
        val_epoch_loss = 0
        for batch in val_dataloader:
            noisy_val_batch, clean_val_batch = batch
            val_pred_clean = model(noisy_val_batch)
            val_loss = loss_fn(val_pred_clean, clean_val_batch)
            val_epoch_loss += val_loss.item()
        avg_val_loss = val_epoch_loss / len(val_dataloader)
        print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    model.train()    

## 7. Save Model Weights

In [ ]:
torch.save(model.state_dict(), 'denoising_cnn_dilated_4layers_4epochs.pth')

## 8. Inference

Test on unseen audio. Result: phase artifacts present (metallic sound) — MSELoss does not account for perceptual audio quality or phase. Next step: perceptual loss or spectral domain approach.

In [ ]:
test_noisy, _ = lb.load('data/archive/noisy_testset_wav/p232_019.wav', sr=None)
test_noisy_tensor = torch.from_numpy(test_noisy).unsqueeze(0)
# run inference
pred_clean = model(test_noisy_tensor).detach().numpy().squeeze(0)


In [ ]:
Audio(test_noisy, rate=16000)


In [ ]:
Audio(pred_clean, rate=16000)
